IMPORT IMPORTANT LIB

In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [3]:
df = pd.read_csv("../dataset/Loan_default.csv")

REMOVING LOANID AGAIN BECOAUSE NEW FRESH NOTBOOK

In [4]:
df.drop("LoanID", axis=1, inplace=True)

In [5]:
X = df.drop("Default", axis=1)
y = df["Default"]

In [6]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}

categorical_cols = X.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

FIRST MODEL (LOGISTIC REGRESSION)

In [9]:
log_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

In [10]:
log_model.fit(X_train, y_train)

c:\Users\DC\loan default prediction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


LogisticRegression(max_iter=1000, random_state=42)

In [11]:
y_pred = log_model.predict(X_test)

In [12]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Accuracy : 0.8850597219502644
Precision: 0.6020066889632107
Recall   : 0.030349013657056147
F1 Score : 0.05778491171749599


In [13]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[45020   119]
 [ 5751   180]]


In [14]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     45139
           1       0.60      0.03      0.06      5931

    accuracy                           0.89     51070
   macro avg       0.74      0.51      0.50     51070
weighted avg       0.85      0.89      0.84     51070



SECOND EXPERIMENT - LOGISTIC REGRESSION WITH CLASS WEIGHT

In [15]:
log_balanced = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight="balanced"
)

In [16]:
log_balanced.fit(X_train, y_train)

c:\Users\DC\loan default prediction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [17]:
y_pred_balanced = log_balanced.predict(X_test)

In [18]:
accuracy_balanced = accuracy_score(y_test, y_pred_balanced)
precision_balanced = precision_score(y_test, y_pred_balanced)
recall_balanced = recall_score(y_test, y_pred_balanced)
f1_balanced = f1_score(y_test, y_pred_balanced)

print("Balanced Logistic Regression")
print("Accuracy :", accuracy_balanced)
print("Precision:", precision_balanced)
print("Recall   :", recall_balanced)
print("F1 Score :", f1_balanced)

Balanced Logistic Regression
Accuracy : 0.673996475425886
Precision: 0.21880575086577816
Recall   : 0.7030854830551341
F1 Score : 0.3337468486133899


In [19]:
cm_balanced = confusion_matrix(y_test, y_pred_balanced)

print("Confusion Matrix:")
print(cm_balanced)

Confusion Matrix:
[[30251 14888]
 [ 1761  4170]]


In [20]:
print("Classification Report:")
print(classification_report(y_test, y_pred_balanced))

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.67      0.78     45139
           1       0.22      0.70      0.33      5931

    accuracy                           0.67     51070
   macro avg       0.58      0.69      0.56     51070
weighted avg       0.86      0.67      0.73     51070



 THIRD EXPERIMENT - THRESHOLD TUNING

In [21]:
# Get probability of Default (Class 1)
y_prob = log_balanced.predict_proba(X_test)[:, 1]

print(y_prob[:10])

[0.18915992 0.16625811 0.3404981  0.60253853 0.44787949 0.07104604
 0.80846827 0.34963673 0.80994562 0.54353805]


In [22]:
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

results = []

for threshold in thresholds:
    y_pred_threshold = (y_prob >= threshold).astype(int)

    precision = precision_score(y_test, y_pred_threshold)
    recall = recall_score(y_test, y_pred_threshold)
    f1 = f1_score(y_test, y_pred_threshold)
    accuracy = accuracy_score(y_test, y_pred_threshold)

    results.append({
        "Threshold": threshold,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

threshold_results = pd.DataFrame(results)

threshold_results

,Threshold,Accuracy,Precision,Recall,F1 Score
0,0.30,0.397552,0.152157,0.915866,0.260959
1,0.35,0.470942,0.164610,0.872534,0.276968
2,0.40,0.545252,0.180717,0.825156,0.296498
3,0.45,0.611768,0.198097,0.768673,0.315011
4,0.50,0.673996,0.218806,0.703085,0.333747
5,0.55,0.728725,0.241905,0.626033,0.348966
6,0.60,0.775250,0.268199,0.541055,0.358628
7,0.65,0.814940,0.299133,0.441915,0.356769
8,0.70,0.844684,0.335309,0.343450,0.339330


In [23]:
best_threshold = 0.60

y_pred_threshold = (y_prob >= best_threshold).astype(int)

print("Threshold:", best_threshold)

print("Accuracy :", accuracy_score(y_test, y_pred_threshold))
print("Precision:", precision_score(y_test, y_pred_threshold))
print("Recall   :", recall_score(y_test, y_pred_threshold))
print("F1 Score :", f1_score(y_test, y_pred_threshold))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_threshold))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_threshold))

Threshold: 0.6
Accuracy : 0.7752496573330723
Precision: 0.26819891349770164
Recall   : 0.5410554712527399
F1 Score : 0.3586276262852034

Confusion Matrix:
[[36383  8756]
 [ 2722  3209]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.81      0.86     45139
           1       0.27      0.54      0.36      5931

    accuracy                           0.78     51070
   macro avg       0.60      0.67      0.61     51070
weighted avg       0.85      0.78      0.81     51070



 MODEL 2 - DECISION TREE

In [28]:
dt_model = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced"
)

In [29]:
dt_model.fit(X_train, y_train)

DecisionTreeClassifier(class_weight='balanced', random_state=42)

In [30]:
dt_model.fit(X_train, y_train)

DecisionTreeClassifier(class_weight='balanced', random_state=42)

In [31]:
y_pred_dt = dt_model.predict(X_test)

In [32]:
accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)

print("Decision Tree")
print("Accuracy :", accuracy_dt)
print("Precision:", precision_dt)
print("Recall   :", recall_dt)
print("F1 Score :", f1_dt)

Decision Tree
Accuracy : 0.8177403563736049
Precision: 0.20465278992478572
Recall   : 0.19726858877086495
F1 Score : 0.20089285714285715


In [33]:
cm_dt = confusion_matrix(y_test, y_pred_dt)

print("Confusion Matrix:")
print(cm_dt)

Confusion Matrix:
[[40592  4547]
 [ 4761  1170]]


In [34]:
print("Classification Report:")
print(classification_report(y_test, y_pred_dt))

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.90      0.90     45139
           1       0.20      0.20      0.20      5931

    accuracy                           0.82     51070
   macro avg       0.55      0.55      0.55     51070
weighted avg       0.81      0.82      0.82     51070

